# Link Budgets

This notebook turns propagation and gain into a practical calculator. It answers the engineering question: given a transmitter, antennas, path, and receiver sensitivity, is the link likely to work?

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Budget Arithmetic

In dB form, a basic link budget is:

`Pr = Pt + Gt + Gr - FSPL - losses`

Compare received power against sensitivity to get margin.

In [ ]:
presets = {
    "Baofeng to Baofeng": dict(tx_power_w=5, tx_gain_dbi=0, rx_gain_dbi=0, freq_mhz=462, distance_km=3, sensitivity_dbm=-118),
    "Baofeng to repeater": dict(tx_power_w=5, tx_gain_dbi=0, rx_gain_dbi=6, freq_mhz=462, distance_km=20, sensitivity_dbm=-120),
    "Meshtastic node to node": dict(tx_power_w=1, tx_gain_dbi=2, rx_gain_dbi=2, freq_mhz=915, distance_km=5, sensitivity_dbm=-130),
}

def watts_to_dbm(watts):
    return 10 * np.log10(watts * 1000)

out = widgets.Output()

def update_budget(preset="Baofeng to Baofeng", tx_power_w=5.0, tx_gain_dbi=0.0, rx_gain_dbi=0.0, freq_mhz=462.0, distance_km=3.0, sensitivity_dbm=-118.0):
    if preset in presets:
        vals = presets[preset]
        tx_power_w = vals["tx_power_w"]
        tx_gain_dbi = vals["tx_gain_dbi"]
        rx_gain_dbi = vals["rx_gain_dbi"]
        freq_mhz = vals["freq_mhz"]
        distance_km = vals["distance_km"]
        sensitivity_dbm = vals["sensitivity_dbm"]
    tx_dbm = watts_to_dbm(tx_power_w)
    path_loss = fspl_db(distance_km, freq_mhz)
    rx_dbm = tx_dbm + tx_gain_dbi + rx_gain_dbi - path_loss
    margin = rx_dbm - sensitivity_dbm
    verdict = "Likely workable" if margin > 10 else "Marginal" if margin > 0 else "Unlikely"
    with out:
        out.clear_output(wait=True)
        print(f"Tx power: {tx_dbm:.2f} dBm")
        print(f"Path loss: {path_loss:.2f} dB")
        print(f"Received power: {rx_dbm:.2f} dBm")
        print(f"Margin above sensitivity: {margin:.2f} dB")
        print(f"Verdict: {verdict}")

controls = widgets.interactive(
    update_budget,
    preset=dropdown(options=list(presets.keys()), value="Baofeng to Baofeng", description="Preset"),
    tx_power_w=float_slider(min_value=0.1, max_value=50, step=0.1, value=5, description="Tx W"),
    tx_gain_dbi=float_slider(min_value=-5, max_value=15, step=0.5, value=0, description="Tx dBi"),
    rx_gain_dbi=float_slider(min_value=-5, max_value=15, step=0.5, value=0, description="Rx dBi"),
    freq_mhz=float_slider(min_value=30, max_value=2400, step=1, value=462, description="MHz"),
    distance_km=float_slider(min_value=0.1, max_value=100, step=0.1, value=3, description="km"),
    sensitivity_dbm=float_slider(min_value=-140, max_value=-80, step=1, value=-118, description="Sens dBm"),
)
display(controls, out)


## Key Takeaway

Link budgets keep RF decisions honest. You can disagree about terrain assumptions or fading margin, but the arithmetic forces every assumption onto the table.